# HW04 LSTM Helper

Use this notebook for **Part II of HW4**.  
Replace the data-loading section with your own dataset, then run through to produce the three required outputs:
1. Training curves
2. Predictions on train / validation / test sets
3. SHAP memory depth

Refer to `02_LSTM_dynamic_inputs.ipynb` for detailed explanations of every step.

In [1]:
# Google Colab setup (skip if running locally)
!pip install -q numpy pandas matplotlib torch shap

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import shap

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


---
## Part 1 — Load Your Data

**Q7**: Describe your inputs and outputs, and how you split your data into train / val / test.

Your CSV should have one row per day with columns for:
- `date` — in any standard format (YYYY-MM-DD)
- forcing variables (e.g. precipitation, temperature, …)
- target variable (e.g. discharge, groundwater level, …)

In [ ]:
# ── TODO: replace with your own file path and column names ───────────────────
DATA_PATH    = 'your_data.csv'          # path to your CSV
FORCING_COLS = ['prcp', 'tmax', 'tmin'] # forcing / input columns
TARGET_COL   = 'discharge'              # output column
DATE_COL     = 'date'                   # date column

# Static catchment attributes (optional).
# Set to a list of column names to enable EA-LSTM; set to None for pure LSTM.
STATIC_COLS  = None                     # e.g. ['area', 'slope', 'soil_type']

TRAIN_END    = '2010-12-31'             # last day of training period
VAL_END      = '2014-12-31'            # last day of validation period
# test period = everything after VAL_END
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL]).sort_values(DATE_COL)
df = df.dropna(subset=FORCING_COLS + [TARGET_COL]).reset_index(drop=True)

dates      = df[DATE_COL].values
train_mask = dates <= np.datetime64(TRAIN_END)
val_mask   = (dates > np.datetime64(TRAIN_END)) & (dates <= np.datetime64(VAL_END))
test_mask  = dates > np.datetime64(VAL_END)

use_static = STATIC_COLS is not None and len(STATIC_COLS) > 0
print(f'Total days : {len(df)}')
print(f'Train      : {train_mask.sum()}  |  Val: {val_mask.sum()}  |  Test: {test_mask.sum()}')
print(f'Forcing variables ({len(FORCING_COLS)}): {FORCING_COLS}')
print(f'Target: {TARGET_COL}')
print(f'Static attributes: {STATIC_COLS if use_static else "None — using pure LSTM"}')
print(f'Model: {"EA-LSTM" if use_static else "LSTM"}')

In [ ]:
# Normalise forcings and target using training-period statistics
F = df[FORCING_COLS].values.astype(np.float32)
F_mu  = F[train_mask].mean(axis=0)
F_std = F[train_mask].std(axis=0) + 1e-6
F_norm = (F - F_mu) / F_std

y     = df[TARGET_COL].values.astype(np.float32)
y_mu  = y[train_mask].mean()
y_std = y[train_mask].std() + 1e-6
y_norm = (y - y_mu) / y_std

print(f'Forcing shape : {F_norm.shape}  (days × features)')
print(f'Target mean (train): {y_mu:.4f}  std: {y_std:.4f}')

# Normalise static attributes (if provided)
if use_static:
    S = df[STATIC_COLS].values.astype(np.float32)
    S_mu  = S[train_mask].mean(axis=0)
    S_std = S[train_mask].std(axis=0) + 1e-6
    S_norm = (S - S_mu) / S_std          # (days, n_static) — same value each day for single catchment
    print(f'Static shape  : {S_norm.shape}  (days × static_features)')
else:
    S_norm = None

---
## Part 2 — Sliding Window Dataset

In [ ]:
SEQ_LEN = 365  # look-back window in days

class TimeSeriesDataset(Dataset):
    """
    Sliding-window dataset.
    If S_norm is provided each sample returns (x_dyn, x_stat, y),
    otherwise returns (x_dyn, y).
    """
    def __init__(self, F_norm, y_norm, seq_len, mask, S_norm=None):
        self.samples   = []
        self.has_static = S_norm is not None
        for t in range(seq_len - 1, len(y_norm)):
            if mask[t]:
                x_dyn = F_norm[t - seq_len + 1 : t + 1]   # (seq_len, n_features)
                if self.has_static:
                    x_stat = S_norm[t]                     # (n_static,) — static at time t
                    self.samples.append((x_dyn, x_stat, y_norm[t]))
                else:
                    self.samples.append((x_dyn, y_norm[t]))

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i):
        if self.has_static:
            x_dyn, x_stat, y = self.samples[i]
            return torch.tensor(x_dyn), torch.tensor(x_stat), torch.tensor([[y]])
        else:
            x, y = self.samples[i]
            return torch.tensor(x), torch.tensor([[y]])

train_ds = TimeSeriesDataset(F_norm, y_norm, SEQ_LEN, train_mask, S_norm)
val_ds   = TimeSeriesDataset(F_norm, y_norm, SEQ_LEN, val_mask,   S_norm)
test_ds  = TimeSeriesDataset(F_norm, y_norm, SEQ_LEN, test_mask,  S_norm)

print(f'Train sequences: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}')

---
## Part 3 — Model

Uses **EA-LSTM** (Kratzert et al. 2019) when `STATIC_COLS` is set, otherwise falls back to a standard **LSTM**.

- **LSTM** — dynamic forcing inputs only.
- **EA-LSTM** — input gate is computed once from static catchment attributes and reused at every timestep, letting the model adapt its sensitivity to forcing based on catchment characteristics.

In [ ]:
# ── TODO: adjust hidden_size and num_layers if needed ────────────────────────
HIDDEN_SIZE = 64
NUM_LAYERS  = 2
# ─────────────────────────────────────────────────────────────────────────────

# ── Pure LSTM (dynamic inputs only) ──────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2 if num_layers > 1 else 0.0)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


# ── EA-LSTM (dynamic + static inputs) ────────────────────────────────────────
class EALSTMCell(nn.Module):
    """
    Entity-Aware LSTM cell (Kratzert et al. 2019).
    The input gate is computed once from static attributes:
        i_t = σ( W_s · s + b_i )   ← constant across all timesteps
    All other gates follow standard LSTM equations.
    """
    def __init__(self, input_size, hidden_size, static_size):
        super().__init__()
        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)  # forget
        self.W_g = nn.Linear(input_size + hidden_size, hidden_size)  # cell
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)  # output
        self.W_s = nn.Linear(static_size, hidden_size)               # input (static)

    def forward(self, x_t, h, c, i_t):
        hx  = torch.cat([h, x_t], dim=1)
        f_t = torch.sigmoid(self.W_f(hx))
        g_t = torch.tanh(self.W_g(hx))
        o_t = torch.sigmoid(self.W_o(hx))
        c   = f_t * c + i_t * g_t
        h   = o_t * torch.tanh(c)
        return h, c


class EALSTMModel(nn.Module):
    def __init__(self, dynamic_size, static_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell        = EALSTMCell(dynamic_size, hidden_size, static_size)
        self.fc          = nn.Linear(hidden_size, 1)

    def forward(self, x_dyn, x_stat):
        # x_dyn  : (batch, seq_len, dynamic_size)
        # x_stat : (batch, static_size)
        batch, seq_len, _ = x_dyn.shape
        i_t = torch.sigmoid(self.cell.W_s(x_stat))   # input gate — static, computed once
        h = torch.zeros(batch, self.hidden_size, device=x_dyn.device)
        c = torch.zeros(batch, self.hidden_size, device=x_dyn.device)
        for t in range(seq_len):
            h, c = self.cell(x_dyn[:, t, :], h, c, i_t)
        return self.fc(h)


# ── Instantiate the right model ───────────────────────────────────────────────
if use_static:
    model = EALSTMModel(len(FORCING_COLS), len(STATIC_COLS), HIDDEN_SIZE).to(device)
    print(f'Using EA-LSTM  |  dynamic={len(FORCING_COLS)}  static={len(STATIC_COLS)}  hidden={HIDDEN_SIZE}')
else:
    model = LSTMModel(len(FORCING_COLS), HIDDEN_SIZE, NUM_LAYERS).to(device)
    print(f'Using LSTM  |  input={len(FORCING_COLS)}  hidden={HIDDEN_SIZE}  layers={NUM_LAYERS}')

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

---
## Part 4 — Train

In [ ]:
# ── TODO: adjust if needed ────────────────────────────────────────────────────
EPOCHS     = 20
LR         = 1e-3
BATCH_SIZE = 64
# ─────────────────────────────────────────────────────────────────────────────

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

def forward_pass(model, batch):
    """Forward pass for both LSTM (2-tuple) and EA-LSTM (3-tuple) batches."""
    if use_static:
        X_dyn, X_stat, y = batch
        X_dyn, X_stat, y = X_dyn.to(device), X_stat.to(device), y.to(device)
        return model(X_dyn, X_stat), y
    else:
        X, y = batch
        X, y = X.to(device), y.to(device)
        return model(X), y

train_losses, val_losses = [], []
best_val, best_state = np.inf, None

for epoch in range(EPOCHS):
    model.train()
    bl = []
    for batch in train_loader:
        pred, y = forward_pass(model, batch)
        optimizer.zero_grad()
        loss = criterion(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bl.append(loss.item())
    train_losses.append(np.mean(bl))

    model.eval()
    with torch.no_grad():
        vl = np.mean([criterion(*forward_pass(model, batch)).item()
                      for batch in val_loader])
    val_losses.append(vl)
    if vl < best_val:
        best_val = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch+1:3d}/{EPOCHS}  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}')

model.load_state_dict(best_state)

---
## Part 5 — Required Plots
### Q8 — include all three plots in your submission

In [ ]:
# Plot 1: Training curves
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, color='steelblue', lw=1.5, label='Train')
ax.plot(val_losses,   color='tomato',    lw=1.5, ls='--', label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (normalised)')
ax.set_title('Training Curves'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Plot 2: Predictions on train / val / test
def predict_period(mask):
    ds  = TimeSeriesDataset(F_norm, y_norm, SEQ_LEN, mask, S_norm)
    ldr = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0)
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in ldr:
            pred, _ = forward_pass(model, batch)
            preds.append(pred.cpu().numpy())
    y_hat = np.concatenate(preds).flatten() * y_std + y_mu
    y_obs = y_norm[mask][SEQ_LEN - 1:] * y_std + y_mu
    t_idx = np.where(mask)[0][SEQ_LEN - 1:]
    return dates[t_idx], y_obs, y_hat

model_name = 'EA-LSTM' if use_static else 'LSTM'
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)
for ax, mask, label, color in zip(
        axes,
        [train_mask, val_mask, test_mask],
        ['Train', 'Validation', 'Test'],
        ['steelblue', 'orange', 'seagreen']):
    t, obs, sim = predict_period(mask)
    nse = 1 - np.sum((obs - sim)**2) / np.sum((obs - obs.mean())**2)
    ax.plot(t, obs, color='k',     lw=0.8, label='Observed')
    ax.plot(t, sim, color=color,   lw=0.8, label=f'Simulated  NSE={nse:.3f}')
    ax.set_ylabel(TARGET_COL); ax.set_title(label); ax.legend(fontsize=8)
plt.suptitle(f'{model_name} Predictions', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Plot 3: SHAP memory depth
np.random.seed(42)
BG_SIZE   = 50
N_EXPLAIN = 50

bg_idx  = np.random.choice(len(train_ds), BG_SIZE,   replace=False)
te_idx  = np.random.choice(len(test_ds),  N_EXPLAIN, replace=False)

X_bg = torch.stack([train_ds[i][0] for i in bg_idx]).to(device)
X_te = torch.stack([test_ds[i][0]  for i in te_idx]).to(device)

model.eval()
explainer = shap.GradientExplainer(model, X_bg)
shap_raw  = explainer.shap_values(X_te)

sv = np.squeeze(shap_raw)                          # (N, SEQ_LEN, n_features)
if sv.ndim == 3 and sv.shape[1] == len(FORCING_COLS):
    sv = sv.transpose(0, 2, 1)

mean_over_time = np.abs(sv).mean(axis=(0, 2))      # (SEQ_LEN,)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.arange(SEQ_LEN), mean_over_time, color='darkorange', lw=1.5)
ax.set_xlabel('Timestep (day 0 = oldest in window)')
ax.set_ylabel('Mean |SHAP|')
ax.set_title('SHAP Memory Depth — How far back does the LSTM look?')
plt.tight_layout(); plt.show()